In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark

25/07/07 01:38:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
events = spark.read.option("header", "true").csv("/home/iceberg/data/events.csv").withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))
devices = spark.read.option("header","true").csv("/home/iceberg/data/devices.csv")

df = events.join(devices,on="device_id",how="left")
df = df.withColumnsRenamed({'browser_type': 'browser_family', 'os_type': 'os_family'})

df.show(10)

+---------+-----------+--------+--------------------+---+--------------------+-------------------+--------------+---------+-----------+
|device_id|    user_id|referrer|                host|url|          event_time|         event_date|browser_family|os_family|device_type|
+---------+-----------+--------+--------------------+---+--------------------+-------------------+--------------+---------+-----------+
|532630305| 1037710827|    NULL| www.zachwilson.tech|  /|2021-03-08 17:27:...|2021-03-08 00:00:00|         Other|    Other|      Other|
|532630305|  925588856|    NULL|    www.eczachly.com|  /|2021-05-10 11:26:...|2021-05-10 00:00:00|         Other|    Other|      Other|
|532630305|-1180485268|    NULL|admin.zachwilson....|  /|2021-02-17 16:19:...|2021-02-17 00:00:00|         Other|    Other|      Other|
|532630305|-1044833855|    NULL| www.zachwilson.tech|  /|2021-09-24 15:53:...|2021-09-24 00:00:00|         Other|    Other|      Other|
|532630305|  747494706|    NULL| www.zachwilson.

In [4]:
sorted = df.repartition(10, col("event_date"))\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = df.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.show(10)
sortedTwo.show(10)


+----------+-----------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------+---------+------------------+
| device_id|    user_id|            referrer|                host|                 url|          event_time|         event_date|browser_family|os_family|       device_type|
+----------+-----------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------+---------+------------------+
| 532630305| 1129583063|                NULL|admin.zachwilson....|                   /|2021-01-07 09:21:...|2021-01-07 00:00:00|         Other|    Other|             Other|
|1088283544| -648945006|                NULL|    www.eczachly.com|                   /|2021-01-07 02:58:...|2021-01-07 00:00:00|      PetalBot|  Android|Generic Smartphone|
|-158310583|-1871780024|                NULL|    www.eczachly.com|                   /|2021-01-07 04:17:...|2021-01-07 00:00:00|      P

In [ ]:
# .sortWithinPartitions() sorts within partitions, whereas .sort() is a global sort, which is very slow

# Note - exchange is synonymous with Shuffle

In [6]:
sorted = df.repartition(10, col("event_date"))\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = df.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.explain()
sortedTwo.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#17, device_id#18, referrer#19, host#20, url#21, cast(event_time#22 as timestamp) AS event_time#288, event_date#29]
   +- Sort [event_date#29 ASC NULLS FIRST, host#20 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(event_date#29, 10), REPARTITION_BY_NUM, [plan_id=294]
         +- Project [user_id#17, device_id#18, referrer#19, host#20, url#21, event_time#22, date_trunc(day, cast(event_time#22 as timestamp), Some(Etc/UTC)) AS event_date#29]
            +- FileScan csv [user_id#17,device_id#18,referrer#19,host#20,url#21,event_time#22] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/iceberg/data/events.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<user_id:string,device_id:string,referrer:string,host:string,url:string,event_time:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_id#17, device_id#18, referr

In [3]:
%%sql

CREATE DATABASE IF NOT EXISTS bootcamp

++
||
++
++

In [10]:
import sys

In [5]:
print(sys.executable)

/usr/local/bin/python


In [6]:
%%sql
    
DROP TABLE IF EXISTS bootcamp.events

++
||
++
++

In [7]:
%%sql

DROP TABLE IF EXISTS bootcamp.events_sorted

++
||
++
++

In [8]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.events (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));


++
||
++
++

In [9]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_sorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));

++
||
++
++

In [10]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_unsorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (year(event_date));

++
||
++
++

In [11]:
%%sql
DESCRIBE TABLE EXTENDED bootcamp.events

col_name,data_type,comment
url,string,None
referrer,string,None
browser_family,string,None
os_family,string,None
device_family,string,None
host,string,None
event_time,timestamp,None
event_date,date,None
,,
# Partitioning,,


In [12]:
%%sql

select * from bootcamp.events;

url,referrer,browser_family,os_family,device_family,host,event_time,event_date


In [21]:
%%sql
select * from demo.bootcamp.events;

url,referrer,browser_family,os_family,device_family,host,event_time,event_date


In [15]:

start_df = df.repartition(4, col("event_date")).withColumn("event_time", col("event_time").cast("timestamp")) \
    
first_sort_df = start_df.sortWithinPartitions(col("event_date"), col("host"))

start_df.write.mode("overwrite").saveAsTable("bootcamp.events_unsorted")
first_sort_df.write.mode("overwrite").saveAsTable("bootcamp.events_sorted")

In [22]:
%%sql

select * from bootcamp.events_unsorted
    limit 10;

device_id,user_id,referrer,host,url,event_time,event_date,browser_family,os_family,device_type
532630305,747494706,None,www.zachwilson.tech,/,2021-09-26 16:03:17.535000,2021-09-26 00:00:00,Other,Other,Other
532630305,747494706,None,admin.zachwilson.tech,/,2021-02-21 16:08:17.975000,2021-02-21 00:00:00,Other,Other,Other
532630305,-824540328,None,admin.zachwilson.tech,/,2021-09-28 17:23:14.992000,2021-09-28 00:00:00,Other,Other,Other
-733097781,5890109,None,www.zachwilson.tech,/,2023-01-09 11:55:28.032000,2023-01-09 00:00:00,curl,Other,Other
532630305,-707954698,None,admin.zachwilson.tech,/,2021-06-19 05:55:00.559000,2021-06-19 00:00:00,Other,Other,Other
532630305,747494706,None,admin.zachwilson.tech,/,2021-06-19 15:09:19.519000,2021-06-19 00:00:00,Other,Other,Other
-733097781,5890109,None,www.zachwilson.tech,/,2023-01-09 20:10:27.610000,2023-01-09 00:00:00,curl,Other,Other
532630305,696863716,None,admin.zachwilson.tech,/,2023-01-10 04:43:49.204000,2023-01-10 00:00:00,Other,Other,Other
532630305,-1180485268,None,www.eczachly.com,/,2021-06-28 16:51:11.679000,2021-06-28 00:00:00,Other,Other,Other
532630305,-1180485268,None,www.zachwilson.tech,/,2021-06-28 17:34:44.423000,2021-06-28 00:00:00,Other,Other,Other


In [27]:
%%sql

select * from bootcamp.events_sorted
    limit 10;

device_id,user_id,referrer,host,url,event_time,event_date,browser_family,os_family,device_type
532630305,-488618451,None,admin.zachwilson.tech,/,2021-01-12 10:22:17.016000,2021-01-12 00:00:00,Other,Other,Other
589185851,-21136712,None,admin.zachwilson.tech,/,2021-01-12 18:49:28.425000,2021-01-12 00:00:00,Chrome,Linux,Other
589185851,-414920062,None,admin.zachwilson.tech,/,2021-01-12 18:54:30.995000,2021-01-12 00:00:00,Chrome,Linux,Other
589185851,-414920062,None,admin.zachwilson.tech,/,2021-01-12 19:56:56.809000,2021-01-12 00:00:00,Chrome,Linux,Other
589185851,-694958230,None,admin.zachwilson.tech,/,2021-01-12 20:08:15.964000,2021-01-12 00:00:00,Chrome,Linux,Other
-290659081,2105351485,None,www.eczachly.com,/,2021-01-12 04:44:23.791000,2021-01-12 00:00:00,bingbot,Other,Spider
-843023486,-2116612468,None,www.eczachly.com,/,2021-01-12 01:13:53.762000,2021-01-12 00:00:00,Chrome,Mac OS X,Other
-843023486,-2116612468,https://www.eczachly.com/,www.eczachly.com,/blog,2021-01-12 01:13:56.914000,2021-01-12 00:00:00,Chrome,Mac OS X,Other
-843023486,-2116612468,https://www.eczachly.com/blog,www.eczachly.com,/graphs,2021-01-12 01:13:58.899000,2021-01-12 00:00:00,Chrome,Mac OS X,Other
-843023486,-2116612468,https://www.eczachly.com/graphs,www.eczachly.com,/graphs,2021-01-12 01:14:01.017000,2021-01-12 00:00:00,Chrome,Mac OS X,Other


In [24]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM demo.bootcamp.events_sorted.files

UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'unsorted' 
FROM demo.bootcamp.events_unsorted.files





size,num_files,sorted
5441299,4,sorted
5553010,4,unsorted


In [19]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM demo.bootcamp.events.files;

size,num_files
None,0


In [25]:
%%sql
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files FROM demo.bootcamp.events_sorted.files;

size,num_files
5441299,4


In [20]:
%%sql 
SELECT COUNT(1) FROM bootcamp.matches_bucketed.files

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `bootcamp`.`matches_bucketed`.`files` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 21;
'Aggregate [unresolvedalias(count(1), None)]
+- 'UnresolvedRelation [bootcamp, matches_bucketed, files], [], false


count(1)
3665
